# Verification: ported pipeline vs original notebook

**What this does.** Runs the same 200 test queries through two
implementations of the hybrid retrieval pipeline and reports whether
they rank documents identically:

1. **Original** — the pipeline exactly as it appears in
   `running-on-fine-tune-of-bge-rera-bge-dense-bm25.ipynb`, the code that
   produced the competition submission.
2. **Ported** — the re-housed version in `prototype/src/`, embedded below
   from the real source files, which is what the web prototype will run.

**Why.** The port moved the same maths into new files so it could answer
one question at a time and load models from Hugging Face. That
restructuring is where a silent bug would hide — most likely in the
reranker's `max_length`, which the two original notebooks disagree about
(512 for hybrid, 256 for pure reranker).

**Setup required before running:**

- Add the competition dataset as an input (for `documents.csv` and
  `test_queries.csv`).
- Add both fine-tuned models as inputs.
- Set **Accelerator → GPU** (T4 or P100). CPU will take far too long.
- Turn **Internet on** if you want the pip install cell to run.

**Result.** The final cell prints PASS or FAIL. PASS means the port is
faithful and safe to deploy.


In [ ]:
import os
import subprocess
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'rank-bm25',
     'sentence-transformers'],
    check=False,
)

import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('WARNING: no GPU. This will be extremely slow.')


## 1. Load data and models

Paths are auto-detected by searching `/kaggle/input`, because the mount
folder names change depending on how inputs were attached.


In [ ]:
import glob
import pandas as pd


def find_one(pattern, what):
    hits = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    if not hits:
        raise FileNotFoundError(
            f'Could not find {what} ({pattern}) under /kaggle/input. '
            'Did you add it as an input to this notebook?'
        )
    print(f'{what}: {hits[0]}')
    return hits[0]


DOCS_CSV = find_one('documents.csv', 'documents.csv')
DENSE_DIR = find_one('fine_tuned_bge_base_agri', 'dense bi-encoder')
RERANK_DIR = find_one('fine_tuned_bge_reranker', 'cross-encoder reranker')

# test_queries.csv may sit alongside documents.csv or in its own folder.
try:
    TEST_CSV = find_one('test_queries.csv', 'test_queries.csv')
    test = pd.read_csv(TEST_CSV)
except FileNotFoundError:
    TEST_CSV = None
    print('test_queries.csv not found; will fall back to training queries.')

df = pd.read_csv(DOCS_CSV, index_col='document_id')
print(f'corpus: {len(df)} documents')

# Uncomment to shorten the run while debugging.
# QUERY_LIMIT = 20
QUERY_LIMIT = None


## 2. Original pipeline (verbatim from the competition notebook)

In [ ]:
import numpy as np
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from tqdm.auto import tqdm

# --- flattening: identical to the competition notebook ---
def create_search_content(row):
    title = str(row.get('title', ' ')).strip()
    text = str(row.get('text', ' ')).strip()
    source = str(row.get('source', ' ')).strip()
    crop = str(row.get('crop', ' ')).strip()
    country = str(row.get('country', ' ')).strip()
    source_url = str(row.get('source_url', ' ')).strip()
    return f'Crop {crop} | Country {country} | Title {title} | Text {text} | Source {source}'


df['search_text'] = df.apply(create_search_content, axis=1)
docs_ids = df.index.to_list()          # NOTE: ints here
corpus_texts = df['search_text'].tolist()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

orig_dense = SentenceTransformer(DENSE_DIR, device=device)
orig_reranker = CrossEncoder(RERANK_DIR, max_length=512, device=device)

id_to_text = dict(zip(docs_ids, corpus_texts))
num_docs = len(corpus_texts)

tokenized_corpus = [doc.lower().split() for doc in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

print('Encoding corpus with the dense bi-encoder...')
doc_embeddings = orig_dense.encode(
    corpus_texts, batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_tensor=True, device=device,
)


def search_bm25(query_text, top_k=50):
    tokenized_query = query_text.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [docs_ids[i] for i in top_indices]


def search_dense(query_text, top_k=50):
    prefixed_query = f'Represent this sentence for searching relevant passages: {query_text}'
    q_emb = orig_dense.encode(
        prefixed_query, normalize_embeddings=True,
        convert_to_tensor=True, device=device,
    )
    sim_scores = torch.matmul(doc_embeddings, q_emb)
    top_res = torch.topk(sim_scores, k=min(top_k, num_docs))
    indices = top_res.indices.cpu().numpy()
    scores = top_res.values.cpu().numpy()
    dense_scores_dict = {docs_ids[i]: float(scores[idx]) for idx, i in enumerate(indices)}
    return list(dense_scores_dict.keys()), dense_scores_dict


def reciprocal_rank_fusion(bm25_ids, dense_ids, k=60):
    rrf_scores = {}
    for rank, doc_id in enumerate(bm25_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    for rank, doc_id in enumerate(dense_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in sorted_docs]


CANDIDATE_POOL_SIZE = 50
ALPHA = 0.0


def original_rank(q_text):
    bm25_top = search_bm25(q_text, top_k=CANDIDATE_POOL_SIZE)
    dense_top_ids, dense_score_map = search_dense(q_text, top_k=CANDIDATE_POOL_SIZE)
    candidate_ids = reciprocal_rank_fusion(bm25_top, dense_top_ids, k=60)[:CANDIDATE_POOL_SIZE]
    pairs = [[q_text, id_to_text[d]] for d in candidate_ids]
    raw = orig_reranker.predict(pairs, batch_size=32, convert_to_numpy=True)
    norm_reranker = 1.0 / (1.0 + np.exp(-raw))
    blended = []
    for idx, d in enumerate(candidate_ids):
        d_score = dense_score_map.get(d, 0.0)
        blended.append((ALPHA * d_score) + ((1.0 - ALPHA) * norm_reranker[idx]))
    blended = np.array(blended)
    top_5 = np.argsort(blended)[::-1][:5]
    return [candidate_ids[i] for i in top_5]


## 3. Ported pipeline

The cells below write the real `prototype/src/` modules to disk and import
them. The source text is embedded verbatim at generation time, so this
tests the shipped code.


In [ ]:
import pathlib

SRC_ROOT = pathlib.Path('src')
SRC_ROOT.mkdir(exist_ok=True)

SOURCES = {
 "__init__.py": "\"\"\"Team Kinyeti retrieval prototype.\n\nPublic demo of the competition retrieval pipeline. See ``config.py`` for the\nconstants ported from the notebooks, and ``pipeline.py`` for the entry point.\n\"\"\"\n",
 "accelerator.py": "\"\"\"ZeroGPU compatibility shim.\n\nHugging Face's ``@spaces.GPU`` decorator requests a GPU for the duration of the\ndecorated call and releases it afterwards. It is documented as *effect-free* in\nnon-ZeroGPU environments, but the ``spaces`` package is only installed there --\nso importing it unconditionally would break local development.\n\nThis module provides a ``gpu`` decorator that is the real thing on the Space and\na transparent no-op everywhere else, letting one codebase run both places.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport functools\nimport logging\nfrom typing import Any, Callable\n\nlogger = logging.getLogger(__name__)\n\ntry:  # pragma: no cover - environment dependent\n    import spaces\n\n    _HAS_SPACES = True\n    logger.info(\"ZeroGPU detected; @gpu will request accelerator time.\")\nexcept ImportError:  # pragma: no cover - local development\n    spaces = None  # type: ignore[assignment]\n    _HAS_SPACES = False\n    logger.info(\"ZeroGPU unavailable; @gpu is a no-op. Expect slow CPU inference.\")\n\n\ndef _noop_decorator(func: Callable) -> Callable:\n    \"\"\"Pass the wrapped function through, preserving metadata.\"\"\"\n\n    @functools.wraps(func)\n    def wrapper(*args: Any, **kwargs: Any) -> Any:\n        return func(*args, **kwargs)\n\n    wrapper.gpu_disabled = True  # type: ignore[attr-defined]\n    return wrapper\n\n\ndef gpu(*dargs: Any, **dkwargs: Any) -> Callable:\n    \"\"\"Request GPU time for a call, or no-op when ZeroGPU is unavailable.\n\n    Supports both spellings used in HF's documentation::\n\n        @gpu\n        def f(): ...\n\n        @gpu(duration=120)\n        def g(): ...\n\n    ``duration`` caps the accelerator runtime in seconds and is passed straight\n    through to ZeroGPU. Shorter durations earn better queue priority, so set it\n    to something realistic rather than leaving the 60s default everywhere.\n    \"\"\"\n    if not _HAS_SPACES:\n        # Called bare (@gpu) -> dargs holds the function itself.\n        if len(dargs) == 1 and callable(dargs[0]) and not dkwargs:\n            return _noop_decorator(dargs[0])\n\n        def decorate(func: Callable) -> Callable:\n            return _noop_decorator(func)\n\n        return decorate\n\n    return spaces.GPU(*dargs, **dkwargs)\n\n\ndef cuda_emulation_active() -> bool:\n    \"\"\"Whether module-level ``.to(\"cuda\")`` is running against emulated CUDA.\n\n    Under ZeroGPU a real GPU only exists inside a ``@gpu`` call, but models are\n    expected to be placed on ``cuda`` at import time. False here means we are on\n    an ordinary machine with no accelerator at all.\n    \"\"\"\n    return _HAS_SPACES\n",
 "bm25.py": "\"\"\"BM25 lexical retrieval.\n\nKept separate from ``retrieve.py`` so the lexical half of the pipeline can be\nbuilt and tested without loading either neural model. On the Space this also\nlets the BM25 index be ready immediately while the weights download.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\n\nimport numpy as np\nfrom rank_bm25 import BM25Okapi\n\nfrom .config import BM25_POOL_SIZE\nfrom .corpus import Corpus\n\nlogger = logging.getLogger(__name__)\n\n\ndef tokenize(text: str) -> list[str]:\n    \"\"\"BM25 tokenisation, matching the notebooks' naive whitespace split.\n\n    The competition notebooks do no stemming or stopword removal. That is\n    reproduced exactly rather than improved, because the fine-tuned reranker was\n    trained on candidates produced by this tokenisation.\n    \"\"\"\n    return text.lower().split()\n\n\nclass BM25Index:\n    \"\"\"Okapi BM25 over the flattened corpus.\"\"\"\n\n    def __init__(self, corpus: Corpus) -> None:\n        self.corpus = corpus\n        self._bm25: BM25Okapi | None = None\n\n    def build(self) -> \"BM25Index\":\n        \"\"\"Build the index. Returns self so it can be chained.\"\"\"\n        logger.info(\"Building BM25 index over %d documents\", len(self.corpus))\n        self._bm25 = BM25Okapi([tokenize(t) for t in self.corpus.search_texts])\n        return self\n\n    @property\n    def is_built(self) -> bool:\n        return self._bm25 is not None\n\n    def search(self, query: str, top_k: int = BM25_POOL_SIZE) -> list[tuple[str, float]]:\n        \"\"\"Return ``(doc_id, score)`` ranked by descending BM25 score.\n\n        ``np.argsort`` ascending then reversed reproduces the notebook, which\n        also means ties resolve by corpus order rather than by score.\n        \"\"\"\n        if self._bm25 is None:\n            raise RuntimeError(\"BM25 index not built; call build() first.\")\n\n        scores = self._bm25.get_scores(tokenize(query))\n        top_indices = np.argsort(scores)[::-1][:top_k]\n        return [(self.corpus.doc_ids[int(i)], float(scores[int(i)])) for i in top_indices]\n",
 "config.py": "\"\"\"Central configuration for the Team Kinyeti retrieval prototype.\n\nConstants here are ported from the competition notebooks in ``scripts/``. Where\nthe notebooks disagree with each other, the disagreement is preserved per-mode\nrather than averaged into a shared default -- see ``RERANKER_MAX_LENGTH``.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nfrom typing import Any, Mapping\n\n# --- Models -----------------------------------------------------------------\n# The Space cannot authenticate against Kaggle, so the fine-tuned weights must\n# live on HF Hub. Override these on the Space via repository variables rather\n# than editing this file, so the same commit runs against a different account.\nDENSE_MODEL_ID = os.getenv(\"DENSE_MODEL_ID\", \"REPLACE-ME/bge-base-agri\")\nRERANKER_MODEL_ID = os.getenv(\"RERANKER_MODEL_ID\", \"REPLACE-ME/bge-reranker-large-agri\")\n\n# --- Document flattening ----------------------------------------------------\n# Identical in all four notebooks. Used for both training and retrieval, which is\n# why it must stay byte-for-byte stable: changing it silently invalidates the\n# fine-tuned models' learned representation.\nFLATTEN_TEMPLATE = \"Crop {crop} | Country {country} | Title {title} | Text {text} | Source {source}\"\n\n# Prepended to *queries only* (never documents) for the dense bi-encoder. This is\n# the instruction format BGE was trained with.\nDENSE_QUERY_PREFIX = \"Represent this sentence for searching relevant passages: \"\n\n# --- First-stage retrieval --------------------------------------------------\nBM25_POOL_SIZE = 50\nDENSE_POOL_SIZE = 50\n\n# Reciprocal rank fusion damping. 60 is the value used in the hybrid notebook.\nRRF_K = 60\n\n# Blend weight between dense cosine similarity (alpha) and sigmoid-normalised\n# reranker logits (1 - alpha). The team grid-searched alpha and found 0.0 best,\n# i.e. pure reranker scores. Kept explicit so the UI can expose it later.\nALPHA = 0.0\n\n# --- Reranker max_length: DELIBERATELY PER-MODE -----------------------------\n# The two inference notebooks disagree, and both are legitimate:\n#\n#   hybrid notebook        -> CrossEncoder(save_dir, max_length=512)\n#   pure-reranker notebook -> InferencePairDataset(..., max_length=256)\n#\n# Training used 256. Unifying these into one shared constant would silently\n# change one submission's rankings, so each mode carries its own value. Do not\n# \"clean this up\".\nRERANKER_MAX_LENGTH = {\n    \"hybrid\": 512,\n    \"pure_reranker\": 256,\n}\n\n# --- Retrieval modes exposed in the UI --------------------------------------\n# `hybrid` and `pure_reranker` reproduce the two competition submissions.\n# `dense_only` and `bm25_only` exist purely as baselines, so a visitor can see\n# what the fine-tuned reranker actually adds.\nMODES = {\n    \"hybrid\": {\n        \"label\": \"Hybrid - BM25 + dense + reranker (best private LB, 0.93753)\",\n        \"description\": (\n            \"Retrieves 50 candidates with BM25 and 50 with the dense bi-encoder, \"\n            \"fuses the two rankings with reciprocal rank fusion, then reranks the \"\n            \"merged pool with the fine-tuned cross-encoder.\"\n        ),\n        \"nDCG@5\": \"0.93753 (private)\",\n    },\n    \"pure_reranker\": {\n        \"label\": \"Pure reranker - scores every document (best public LB, 0.96888)\",\n        \"description\": (\n            \"Skips first-stage retrieval entirely and scores all 695 documents \"\n            \"with the fine-tuned cross-encoder. This was the strongest public \"\n            \"leaderboard submission.\"\n        ),\n        \"nDCG@5\": \"0.96888 (public)\",\n    },\n    \"dense_only\": {\n        \"label\": \"Dense only - semantic similarity baseline\",\n        \"description\": \"Fine-tuned bi-encoder cosine similarity. No lexical matching, no reranking.\",\n        \"nDCG@5\": \"0.825 (held-out)\",\n    },\n    \"bm25_only\": {\n        \"label\": \"BM25 only - keyword baseline\",\n        \"description\": \"Classic lexical search. No semantics, no reranking.\",\n        \"nDCG@5\": \"not reported\",\n    },\n}\n\nDEFAULT_MODE = \"hybrid\"\n\n# --- Corpus -----------------------------------------------------------------\nDOCUMENTS_CSV = os.getenv(\"DOCUMENTS_CSV\", \"data/documents.csv\")\n\n# Result cards shown, matching the competition's submission format.\nTOP_K = 5\n\n\ndef create_search_content(row: Mapping[str, Any]) -> str:\n    \"\"\"Flatten one corpus row into the single searchable string.\n\n    Ported verbatim from ``create_search_content`` in all four notebooks. The\n    odd-looking defaults (a single space rather than empty string) are preserved\n    because they are part of what the models were fine-tuned against.\n    \"\"\"\n    return FLATTEN_TEMPLATE.format(\n        crop=str(row.get(\"crop\", \" \")).strip(),\n        country=str(row.get(\"country\", \" \")).strip(),\n        title=str(row.get(\"title\", \" \")).strip(),\n        text=str(row.get(\"text\", \" \")).strip(),\n        source=str(row.get(\"source\", \" \")).strip(),\n    )\n\n\ndef is_placeholder_models() -> bool:\n    \"\"\"True when the model IDs have not been configured yet.\"\"\"\n    return DENSE_MODEL_ID.startswith(\"REPLACE-ME\") or RERANKER_MODEL_ID.startswith(\"REPLACE-ME\")\n",
 "corpus.py": "\"\"\"Corpus loading and preprocessing.\n\nReplaces the notebooks' hardcoded ``/kaggle/input/...`` reads with a local CSV\npath, keeping the flattening step identical (see ``config.create_search_content``).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nimport os\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom .config import create_search_content\n\nlogger = logging.getLogger(__name__)\n\n# Columns the retrieval pipeline depends on. Missing ones degrade the flattened\n# string rather than raising, matching the notebooks' tolerant `.get(..., \" \")`.\nEXPECTED_COLUMNS = (\"title\", \"text\", \"source\", \"crop\", \"country\", \"source_url\")\n\n\n@dataclass(frozen=True)\nclass Corpus:\n    \"\"\"An immutable, indexed corpus ready for retrieval.\n\n    ``doc_ids`` and ``search_texts`` are parallel lists: index ``i`` in one\n    corresponds to index ``i`` in the other. This ordering is what lets the\n    numpy argsort steps map scored positions back to document IDs, so nothing\n    here should ever be reordered or sorted in place.\n    \"\"\"\n\n    doc_ids: list[str]\n    search_texts: list[str]\n    frame: pd.DataFrame\n\n    def __len__(self) -> int:\n        return len(self.doc_ids)\n\n    def record(self, doc_id: str) -> dict:\n        \"\"\"Return the display fields for one document.\"\"\"\n        row = self.frame.loc[doc_id]\n        return {\n            \"document_id\": doc_id,\n            \"title\": _clean(row.get(\"title\")),\n            \"text\": _clean(row.get(\"text\")),\n            \"source\": _clean(row.get(\"source\")),\n            \"crop\": _clean(row.get(\"crop\")),\n            \"country\": _clean(row.get(\"country\")),\n            \"source_url\": _clean(row.get(\"source_url\")),\n        }\n\n\ndef _clean(value: object) -> str:\n    \"\"\"Normalise a cell to a stripped string, mapping NaN/None to empty.\"\"\"\n    if value is None or (isinstance(value, float) and pd.isna(value)):\n        return \"\"\n    text = str(value).strip()\n    return \"\" if text.lower() in {\"nan\", \"none\"} else text\n\n\n#: Where ``documents.csv`` may live, in priority order.\n#: The repo-root location suits local development (the competition data sits in\n#: the project's own ``data/``); the package-local location is what the Space\n#: needs, since its repo root is ``prototype/``.\n_CANDIDATE_PATHS = (\n    Path(\"data/documents.csv\"),\n    Path(__file__).resolve().parent.parent / \"data\" / \"documents.csv\",\n    Path(__file__).resolve().parent.parent.parent / \"data\" / \"documents.csv\",\n)\n\n\ndef resolve_documents_csv(explicit: str | Path | None = None) -> Path:\n    \"\"\"Find ``documents.csv``, honouring an explicit path or ``DOCUMENTS_CSV``.\n\n    Raises with an actionable message rather than letting pandas emit an opaque\n    ``FileNotFoundError`` from deep inside ``read_csv``.\n    \"\"\"\n    if explicit is not None:\n        candidates = [Path(explicit)]\n    elif os.getenv(\"DOCUMENTS_CSV\"):\n        candidates = [Path(os.environ[\"DOCUMENTS_CSV\"])]\n    else:\n        candidates = list(_CANDIDATE_PATHS)\n\n    for candidate in candidates:\n        if candidate.exists():\n            return candidate\n\n    searched = \"\\n  \".join(str(c) for c in candidates)\n    raise FileNotFoundError(\n        \"Could not find documents.csv. Searched:\\n  \"\n        f\"{searched}\\n\"\n        \"Download it from the competition and place it in the project's data/ \"\n        \"folder, or set the DOCUMENTS_CSV environment variable to its path.\"\n    )\n\n\ndef load_corpus(path: str | Path | None = None) -> Corpus:\n    \"\"\"Load ``documents.csv`` and flatten each row into its search string.\n\n    The CSV is indexed by ``document_id``, matching the notebooks'\n    ``read_csv(..., index_col=\"document_id\")``.\n    \"\"\"\n    csv_path = resolve_documents_csv(path)\n    frame = pd.read_csv(csv_path, index_col=\"document_id\")\n\n    missing = [c for c in EXPECTED_COLUMNS if c not in frame.columns]\n    if missing:\n        logger.warning(\n            \"Corpus is missing expected column(s): %s. Retrieval will still run, \"\n            \"but flattened documents will be weaker than the fine-tuned models expect.\",\n            \", \".join(missing),\n        )\n\n    frame[\"search_text\"] = frame.apply(create_search_content, axis=1)\n\n    # Normalise the index to strings *in place*, so the frame and `doc_ids`\n    # always agree. The CSV stores integer IDs, but submission files and the\n    # trace objects use string IDs; converting only one side made every lookup\n    # by ID fail with a KeyError.\n    frame.index = frame.index.map(str)\n\n    doc_ids = frame.index.to_list()\n    search_texts = frame[\"search_text\"].tolist()\n\n    logger.info(\"Loaded %d documents from %s\", len(doc_ids), csv_path)\n    return Corpus(doc_ids=doc_ids, search_texts=search_texts, frame=frame)\n",
 "retrieve.py": "\"\"\"Model-backed retrieval: dense bi-encoder search and cross-encoder reranking.\n\nPorted from:\n\n* ``scripts/running-on-fine-tune-of-bge-rera-bge-dense-bm25.ipynb``  (hybrid)\n* ``scripts/running-inference-on-fine-tune-of-bge-reranker.ipynb``   (pure reranker)\n\nImports of ``torch`` and ``sentence_transformers`` are deferred into the methods\nthat need them. That keeps module import cheap, so the BM25-only paths and the UI\nshell can start without pulling several gigabytes of wheels, and so the ranking\nmaths in ``scoring.py`` stays testable on its own.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\n\nfrom .accelerator import gpu\nfrom .config import DENSE_POOL_SIZE, DENSE_QUERY_PREFIX\nfrom .corpus import Corpus\n\nlogger = logging.getLogger(__name__)\n\n\nclass Retriever:\n    \"\"\"Holds the two fine-tuned models and the encoded corpus.\n\n    Models are loaded once and reused across queries. On ZeroGPU they are placed\n    on ``cuda`` at load time as HF requires, which works because CUDA emulation\n    is active outside ``@gpu`` calls.\n    \"\"\"\n\n    def __init__(\n        self,\n        corpus: Corpus,\n        dense_model_id: str,\n        reranker_model_id: str,\n        device: str | None = None,\n    ) -> None:\n        import torch\n        from sentence_transformers import CrossEncoder, SentenceTransformer\n\n        self.corpus = corpus\n        self.dense_model_id = dense_model_id\n        self.reranker_model_id = reranker_model_id\n        self.device = device or (\"cuda\" if torch.cuda.is_available() else \"cpu\")\n\n        logger.info(\"Loading dense bi-encoder %s on %s\", dense_model_id, self.device)\n        self.dense_model = SentenceTransformer(dense_model_id, device=self.device)\n\n        logger.info(\"Loading cross-encoder %s on %s\", reranker_model_id, self.device)\n        # max_length is set per call, not here: the two source notebooks use\n        # different values and baking one in would corrupt the other's rankings.\n        self.reranker = CrossEncoder(reranker_model_id, device=self.device)\n\n        self._doc_embeddings = None\n\n    # -- indexing -------------------------------------------------------------\n\n    @property\n    def is_encoded(self) -> bool:\n        return self._doc_embeddings is not None\n\n    @gpu(duration=120)\n    def encode_corpus(self) -> None:\n        \"\"\"Pre-encode every document with the fine-tuned bi-encoder.\n\n        Done once. ``normalize_embeddings=True`` so cosine similarity reduces to\n        a dot product, exactly as in the notebook.\n        \"\"\"\n        if self._doc_embeddings is not None:\n            return\n\n        logger.info(\"Encoding %d passages\", len(self.corpus))\n        self._doc_embeddings = self.dense_model.encode(\n            self.corpus.search_texts,\n            batch_size=64,\n            show_progress_bar=False,\n            normalize_embeddings=True,\n            convert_to_tensor=True,\n            device=self.device,\n        )\n\n    # -- dense retrieval ------------------------------------------------------\n\n    @gpu(duration=60)\n    def _encode_query(self, query_text: str):\n        \"\"\"Encode one query with the BGE instruction prefix.\n\n        The prefix is not cosmetic: the dense model was fine-tuned with it, so\n        dropping it degrades retrieval.\n        \"\"\"\n        return self.dense_model.encode(\n            f\"{DENSE_QUERY_PREFIX}{query_text}\",\n            normalize_embeddings=True,\n            convert_to_tensor=True,\n            device=self.device,\n        )\n\n    def search_dense(self, query_text: str, top_k: int = DENSE_POOL_SIZE) -> list[tuple[str, float]]:\n        \"\"\"Semantic retrieval by cosine similarity over the encoded corpus.\"\"\"\n        import torch\n\n        if self._doc_embeddings is None:\n            raise RuntimeError(\"Corpus not encoded; call encode_corpus() first.\")\n\n        query_embedding = self._encode_query(query_text)\n        # Both sides are L2-normalised, so the dot product is the cosine.\n        similarities = torch.matmul(self._doc_embeddings, query_embedding)\n        top = torch.topk(similarities, k=min(top_k, len(self.corpus)))\n\n        indices = top.indices.cpu().numpy()\n        values = top.values.cpu().numpy()\n        return [\n            (self.corpus.doc_ids[int(i)], float(values[n]))\n            for n, i in enumerate(indices)\n        ]\n\n    # -- reranking ------------------------------------------------------------\n\n    @gpu(duration=120)\n    def rerank(\n        self,\n        query_text: str,\n        doc_ids: list[str],\n        max_length: int,\n    ) -> list[tuple[str, float]]:\n        \"\"\"Score ``(query, document)`` pairs with the fine-tuned cross-encoder.\n\n        Returns raw, unbounded logits; sigmoid normalisation happens in the\n        blending step, matching the notebook.\n\n        ``max_length`` is required rather than defaulted. The hybrid submission\n        uses 512 and the pure-reranker submission uses 256, so a shared default\n        would silently change one of them.\n        \"\"\"\n        id_to_text = dict(zip(self.corpus.doc_ids, self.corpus.search_texts))\n\n        # The reranker is constructed without max_length so it can vary per call.\n        self.reranker.max_length = max_length\n\n        pairs = [[query_text, id_to_text[doc_id]] for doc_id in doc_ids]\n        logits = self.reranker.predict(pairs, batch_size=32, convert_to_numpy=True)\n        return [(doc_id, float(v)) for doc_id, v in zip(doc_ids, logits)]\n",
 "scoring.py": "\"\"\"Lightweight ranking maths: fusion, blending, and result types.\n\nDeliberately free of torch / sentence-transformers imports so the ranking logic\ncan be exercised without downloading any model weights. See ``bm25.py`` for the\nlexical index and ``retrieve.py`` for the model-backed retrievers.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass, field\n\nfrom .config import ALPHA, RRF_K\n\n\n@dataclass\nclass Scored:\n    \"\"\"One document carrying the evidence of how it was retrieved.\n\n    The per-stage fields are the point of the UI: showing BM25 rank, dense rank,\n    RRF rank and the reranker's score side by side is what makes the fine-tuned\n    reranker's contribution visible to a reader who has never heard of nDCG.\n    \"\"\"\n\n    doc_id: str\n    bm25_rank: int | None = None\n    bm25_score: float | None = None\n    dense_rank: int | None = None\n    dense_score: float | None = None\n    rrf_rank: int | None = None\n    rrf_score: float | None = None\n    reranker_logit: float | None = None\n    reranker_score: float | None = None\n    final_rank: int | None = None\n\n    @property\n    def retrieved_by(self) -> list[str]:\n        \"\"\"Which first-stage retrievers surfaced this document.\"\"\"\n        found = []\n        if self.bm25_rank is not None:\n            found.append(\"BM25\")\n        if self.dense_rank is not None:\n            found.append(\"dense\")\n        return found\n\n\n@dataclass\nclass Timings:\n    \"\"\"Milliseconds spent per stage, surfaced in the UI.\"\"\"\n\n    bm25_ms: float = 0.0\n    dense_ms: float = 0.0\n    rerank_ms: float = 0.0\n    total_ms: float = 0.0\n\n    def as_dict(self) -> dict[str, float]:\n        return {\n            \"BM25\": round(self.bm25_ms, 1),\n            \"dense\": round(self.dense_ms, 1),\n            \"rerank\": round(self.rerank_ms, 1),\n            \"total\": round(self.total_ms, 1),\n        }\n\n\n@dataclass\nclass RetrievalResult:\n    \"\"\"Everything one query produced, including the full audit trail.\"\"\"\n\n    query: str\n    mode: str\n    final: list[Scored] = field(default_factory=list)\n    #: Every candidate that entered the final stage, ranked. The UI uses this to\n    #: show what the reranker demoted, not just what it promoted.\n    pool: list[Scored] = field(default_factory=list)\n    timings: Timings = field(default_factory=Timings)\n    notes: list[str] = field(default_factory=list)\n\n\ndef reciprocal_rank_fusion(\n    bm25_ids: list[str],\n    dense_ids: list[str],\n    k: int = RRF_K,\n) -> list[tuple[str, float]]:\n    \"\"\"Fuse two ranked lists by reciprocal rank.\n\n    Scores by rank *position* rather than raw score, which is what lets BM25's\n    unbounded term-frequency scores and cosine similarities be combined without\n    first putting them on a common scale.\n\n    Ported from ``reciprocal_rank_fusion`` in the hybrid notebook.\n    \"\"\"\n    rrf_scores: dict[str, float] = {}\n    for ranked_list in (bm25_ids, dense_ids):\n        for rank, doc_id in enumerate(ranked_list):\n            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)\n\n    return sorted(rrf_scores.items(), key=lambda kv: kv[1], reverse=True)\n\n\ndef sigmoid(x: float) -> float:\n    \"\"\"Logistic squashing, as used in the notebook's alpha-blending step.\n\n    Guarded against overflow: reranker logits are unbounded and this is called\n    per candidate in a loop.\n    \"\"\"\n    if x >= 0:\n        return 1.0 / (1.0 + math.exp(-x))\n    exp_x = math.exp(x)\n    return exp_x / (1.0 + exp_x)\n\n\ndef blend_scores(\n    reranker_logits: list[float],\n    dense_scores: list[float],\n    alpha: float = ALPHA,\n) -> list[float]:\n    \"\"\"Blend normalised reranker scores with dense similarities.\n\n    At the competition's alpha of 0.0 this reduces to the pure reranker score,\n    but the parameter is retained so the grid-searched alternatives\n    (0.2 / 0.4 / 0.5 / 0.6 / 0.8 / 1.0) can be explored interactively.\n\n    Documents that arrived via BM25 only have no dense similarity; the notebook\n    defaults those to 0.0, faithfully reproduced here.\n    \"\"\"\n    return [\n        (alpha * dense) + ((1.0 - alpha) * sigmoid(logit))\n        for logit, dense in zip(reranker_logits, dense_scores)\n    ]\n"
}

for _name, _body in SOURCES.items():
    (SRC_ROOT / _name).write_text(_body, encoding='utf-8')
    print(f'wrote src/{_name} ({len(_body.splitlines())} lines)')


Now import the ported modules and build the index.

In [ ]:
import importlib
import sys

sys.path.insert(0, '.')
for _name in list(sys.modules):
    if _name == 'src' or _name.startswith('src.'):
        del sys.modules[_name]

from src.config import RERANKER_MAX_LENGTH
from src.corpus import load_corpus
from src.bm25 import BM25Index
from src.retrieve import Retriever
from src.scoring import blend_scores, reciprocal_rank_fusion as ported_rrf

# The ported corpus loader would look for a local documents.csv; point it
# at the Kaggle path explicitly.
import os
os.environ['DOCUMENTS_CSV'] = DOCS_CSV
corpus = load_corpus(DOCS_CSV)
print(f'ported corpus: {len(corpus)} documents')
print(f"reranker max_length by mode: {RERANKER_MAX_LENGTH}")

ported_bm25 = BM25Index(corpus).build()
retriever = Retriever(corpus, DENSE_DIR, RERANK_DIR, device=device)
retriever.encode_corpus()
print('ported index ready')


In [ ]:
def ported_rank(q_text):
    bm25_hits = ported_bm25.search(q_text, top_k=50)
    dense_hits = retriever.search_dense(q_text, top_k=50)

    bm25_ids = [d for d, _ in bm25_hits]
    dense_ids = [d for d, _ in dense_hits]
    dense_scores = dict(dense_hits)

    fused = ported_rrf(bm25_ids, dense_ids, k=60)[:50]
    candidate_ids = [d for d, _ in fused]

    logits = retriever.rerank(
        q_text, candidate_ids, max_length=RERANKER_MAX_LENGTH['hybrid']
    )
    rerank_scores = dict(logits)

    blended = blend_scores(
        [rerank_scores[d] for d in candidate_ids],
        [dense_scores.get(d, 0.0) for d in candidate_ids],
        alpha=0.0,
    )
    order = np.argsort(np.array(blended))[::-1][:5]
    return [candidate_ids[i] for i in order]


## 4. Compare

Document IDs are compared as strings: the original notebook keeps integer
IDs while the port uses strings, which is a representation difference and
not a ranking difference.


In [ ]:
if TEST_CSV is None:
    # Fall back to the corpus titles as pseudo-queries so the cell still runs.
    queries = df['title'].dropna().astype(str).head(50).tolist()
    query_ids = [f'q{i}' for i in range(len(queries))]
    print('Using corpus titles as stand-in queries.')
else:
    qid_col = 'QueryId' if 'QueryId' in test.columns else test.columns[0]
    qtext_col = 'Query' if 'Query' in test.columns else test.columns[1]
    subset = test if QUERY_LIMIT is None else test.head(QUERY_LIMIT)
    query_ids = [str(v).replace('.0', '') for v in subset[qid_col]]
    queries = [str(v).strip() for v in subset[qtext_col]]

print(f'Comparing {len(queries)} queries...')

mismatches = []
for qid, qtext in tqdm(list(zip(query_ids, queries))):
    try:
        a = [str(x) for x in original_rank(qtext)]
        b = [str(x) for x in ported_rank(qtext)]
    except Exception as exc:
        mismatches.append((qid, qtext, f'ERROR: {exc}', '', ''))
        continue
    if a != b:
        mismatches.append((qid, qtext, a, b, len(set(a) & set(b))))

print()
print('=' * 70)
if not mismatches:
    print(f'PASS - all {len(queries)} queries ranked identically.')
    print('The port is faithful; safe to deploy.')
else:
    print(f'FAIL - {len(mismatches)} of {len(queries)} queries differ.')
    print('=' * 70)
    for qid, qtext, a, b, overlap in mismatches[:10]:
        print(f'\nQuery {qid}: {qtext}')
        print(f'  original: {a}')
        print(f'  ported:   {b}')
        if overlap != '':
            print(f'  shared documents: {overlap}/5')
print('=' * 70)


## Reading the result

- **PASS** — the port ranks identically. Deploy it.
- **FAIL with high overlap** (4/5 shared, different order) — a tie-breaking
  or floating-point difference. Usually harmless, but worth a look.
- **FAIL with low overlap** — a real bug. The `max_length` setting is the
  first thing to check, since the hybrid and pure-reranker notebooks use
  different values.
